In [1]:
import os
import pyarrow as pa
from pathlib import Path
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import TimeSeriesSplit
from tqdm import tqdm
from joblib import Parallel, delayed
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
_NCPU = os.cpu_count() or 1
pa.set_cpu_count(_NCPU)
pa.set_io_thread_count(_NCPU)
os.environ["NUMEXPR_MAX_THREADS"] = str(_NCPU)
os.environ["NUMEXPR_NUM_THREADS"] = str(_NCPU)
os.environ.setdefault("OMP_NUM_THREADS", str(_NCPU))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(_NCPU))
os.environ.setdefault("MKL_NUM_THREADS", str(_NCPU))
print(f"Running with {_NCPU} CPU cores | pyarrow {pa.__version__}", flush=True)

Running with 8 CPU cores | pyarrow 24.0.0


Variables

In [2]:
# LOAD DATA
TARGET_COLS = {"nsw_price", "qld_price", "sa_price", "vic_price", "price"}
FEATURE_DATASET_START = pd.Timestamp("2019/01/01")
FEATIRE_DATASET_END = pd.Timestamp("2026/01/01")
os.environ["FEATURE_DATASET"] = "1_dispatch_price.parquet"
os.environ["TARGET"] = "NSW" 
os.environ["TARGETS_PATH"] = f"../3_Targets build/Target_data/{os.environ["TARGET"].lower()}_targets.parquet"
os.environ["FEATURES_PATH"] = f"../2_Features build/Feature_data/{os.environ['FEATURE_DATASET']}"
os.environ["features_output"] = f"../4_Features select/Feature_store/{os.environ["TARGET"]}_features_{os.environ['FEATURE_DATASET']}"
os.environ["future_prediction_targets_output"] = f"../4_Features select/Feature_store/{os.environ["TARGET"]}_targets_{os.environ['FEATURE_DATASET']}"

# AGGREGATE TARGETS
os.environ["OUTPUT_RESOLUTION"] = "30"
os.environ["future_prediction_targets_agg_output"] = f"../4_Features select/Feature_store/{os.environ["TARGET"]}_targets_agg_{os.environ['FEATURE_DATASET']}"

# FEATURE RANKING
FEATURE_SELECTION_SUBSAMPLE_START = pd.Timestamp("2019/01/01")
FEATURE_SELECTION_SUBSAMPLE_END = pd.Timestamp("2023/01/01")
FEATURE_RANKING_SUBSAMPLE_AMOUNT = 1_000
os.environ["feature_data_output"] = f"../4_Features select/Feature_store/{os.environ["TARGET"]}_feature_data_{os.environ['FEATURE_DATASET']}"

# ADD WINDOWS
os.environ["HORIZONS_PER_WINDOW"] = "8"       

# REMOVE DUPLICATE FEATURES
DEDUP_SUBSAMPLE_AMOUNT = 500_000
os.environ["feature_data_unique_output"] = f"../4_Features select/Feature_store/{os.environ["TARGET"]}_feature_data_unique_{os.environ['FEATURE_DATASET']}"

# FEATURE SELECTION CV
FEATURE_SELECTION_CV_SUBSAMPLE_START = pd.Timestamp("2019/01/01")
FEATURE_SELECTION_CV_SUBSAMPLE_END = pd.Timestamp("2023/01/01")
FEATURE_SELECTION_CV_SUBSAMPLE_AMOUNT = 50_000

# FEATURE SELECTION OUTPUT
os.environ["cv_selection_output"] = f"../4_Features select/Feature_store/{os.environ['TARGET']}_cv_selection_{os.environ['FEATURE_DATASET']}"
os.environ["selected_features_output"] = f"../4_Features select/Selected_features/{os.environ['TARGET']}_selected_features_{os.environ['FEATURE_DATASET']}"


Load raw data

In [3]:
if False:
    def load_data():
        def load_features(
                feature_dataset_start: pd.Timestamp,
                feature_dataset_end: pd.Timestamp,
        ) -> pd.DataFrame:
            features = pd.read_parquet(os.environ["FEATURES_PATH"], filters=[
                ('SETTLEMENTDATE', '>=', feature_dataset_start),
                ('SETTLEMENTDATE', '<=', feature_dataset_end),
            ])
            
            features = features.drop(columns=[c for c in features.columns if c in TARGET_COLS])
            return features.loc[:feature_dataset_end]


        def load_targets(target:str):
            future_prediction_targets = pd.read_parquet(os.environ["TARGETS_PATH"] )
            return future_prediction_targets


        # Make calls
        features = load_features(
            feature_dataset_start=FEATURE_DATASET_START,
            feature_dataset_end=FEATIRE_DATASET_END,
        )

        future_prediction_targets = load_targets(
            target=os.environ["TARGET"]
        )

        # Align on index
        future_prediction_targets = future_prediction_targets.loc[features.index]
        features = features.loc[future_prediction_targets.index]

        
        display(features[:3])
        display(future_prediction_targets[:3])

         
        features.to_parquet(os.environ["features_output"])
        future_prediction_targets.to_parquet(os.environ["future_prediction_targets_output"])
    
    load_data()

Aggregate targets to output resolution

In [4]:
if False:
    def aggregate_targets(future_prediction_targets):
        OUTPUT_RESOLUTION = int(os.environ["OUTPUT_RESOLUTION"])

        target_cols = list(future_prediction_targets.columns)
        horizons = len(target_cols)
        bucket_size = max(1, OUTPUT_RESOLUTION // 5)
        n_buckets = horizons // bucket_size
        n_raw = n_buckets * bucket_size

        values = future_prediction_targets.values.astype(np.float32)
        aggregated = values[:, :n_raw].reshape(len(values), n_buckets, bucket_size).mean(axis=2)
        bucket_cols = [f"h{i + 1}" for i in range(n_buckets)]

        future_prediction_targets_agg = pd.DataFrame(
            aggregated,
            index=future_prediction_targets.index,
            columns=bucket_cols,
        )
        display(future_prediction_targets_agg[:3])

        future_prediction_targets_agg.to_parquet(os.environ["future_prediction_targets_agg_output"])
    
    
    future_prediction_targets = pd.read_parquet(os.environ["future_prediction_targets_output"])
    aggregate_targets(future_prediction_targets)

Feature ranking

In [5]:
if False:
    """
    Rank features

    MI ranking: measures mutual information — a non-linear, information-theoretic score. It answers "does knowing this feature reduce uncertainty about the target?",
    capturing non-linear dependencies too. Targets are passed in pre-aggregated to output_resolution_minutes,
    so MI is computed directly at the output resolution.
    """

    def rank_features(features:pd.DataFrame,
        future_prediction_targets_agg: pd.DataFrame,
        feature_selection_subsample_start: pd.Timestamp,
        feature_selection_subsample_end: pd.Timestamp,
        feature_selection_subsample_amount: int,
        output_resolution_minutes: int,
    ):
        
        target_rounds = 10

        feature_cols = list(features.columns)
        target_cols_agg = list(future_prediction_targets_agg.columns)
        n_buckets = len(target_cols_agg)

        def _filter_data_for_feature_time_range_subset():
            features_subset = features.loc[feature_selection_subsample_start:feature_selection_subsample_end]
            targets_subset = future_prediction_targets_agg.loc[feature_selection_subsample_start:feature_selection_subsample_end]
            shared_index = features_subset.index.intersection(targets_subset.index)
            return features_subset.loc[shared_index], targets_subset.loc[shared_index]

        features_subset, targets_subset = _filter_data_for_feature_time_range_subset()

        features_subset = features_subset.values.astype(np.float32)
        targets_subset = targets_subset.reset_index(drop=True)

        def _subsample_features():
            seed = np.random.default_rng(42)
            n_samples = min(feature_selection_subsample_amount, len(features_subset))
            subsample_index = seed.choice(len(features_subset), size=n_samples, replace=False)
            subsample_index.sort()
            return features_subset[subsample_index], targets_subset.iloc[subsample_index]

        X_subsample, y_subsample = _subsample_features()

        def _mutual_information_scoring():
            aggregated_target_matrix = y_subsample[target_cols_agg].values.astype(np.float32)
            n_features = X_subsample.shape[1]
            n_subsample_rows = X_subsample.shape[0]

            # Split features into chunks so n_tasks >> n_workers → multiple rounds
            n_feature_chunks = max(1, (target_rounds * _NCPU + n_buckets - 1) // n_buckets)
            chunk_edges = np.array_split(np.arange(n_features), n_feature_chunks)
            feature_chunk_ranges = [(int(c[0]), int(c[-1]) + 1) for c in chunk_edges if len(c) > 0]
            n_tasks = n_buckets * len(feature_chunk_ranges)

            print(
                f"  MI scoring: {n_features} features × {n_buckets} horizons ({output_resolution_minutes}-min) "
                f"| subsample n={n_subsample_rows:,} | {n_tasks} tasks across {_NCPU} CPUs (~{n_tasks // _NCPU} rounds)",
                flush=True,
            )

            def _score_chunk(bucket_idx, feature_start_idx, feature_end_idx):
                return bucket_idx, feature_start_idx, feature_end_idx, mutual_info_regression(
                    X_subsample[:, feature_start_idx:feature_end_idx], aggregated_target_matrix[:, bucket_idx],
                    discrete_features=False, n_neighbors=3, random_state=42,
                )

            gen = Parallel(n_jobs=-1, backend="loky", batch_size=1, return_as="generator_unordered")(
                delayed(_score_chunk)(bucket_idx, feature_start_idx, feature_end_idx)
                for bucket_idx in range(n_buckets) for feature_start_idx, feature_end_idx in feature_chunk_ranges
            )
            scores = np.empty((n_features, n_buckets))
            for bucket_idx, feature_start_idx, feature_end_idx, chunk_scores in tqdm(gen, total=n_tasks, desc="MI scoring", leave=True):
                scores[feature_start_idx:feature_end_idx, bucket_idx] = chunk_scores

            mi_matrix = pd.DataFrame(scores, index=feature_cols, columns=target_cols_agg)
            feature_cols_ranked = mi_matrix.mean(axis=1).sort_values(ascending=False)
            return feature_cols_ranked, mi_matrix

        feature_cols_ranked, mi_matrix = _mutual_information_scoring()

        # Save to parquet
        df = pd.DataFrame({
            "rank": range(1, len(feature_cols_ranked) + 1),
            "feature": feature_cols_ranked.index,
            "mean_mi": feature_cols_ranked.values,
            "target": os.environ["TARGET"],
            "feature_dataset": Path(os.environ["FEATURE_DATASET"]).stem,
        }).set_index("feature")

        df_horizons = mi_matrix.reindex(feature_cols_ranked.index)
        feature_data = pd.concat([df, df_horizons], axis=1).reset_index(names="feature")
        feature_data.to_parquet(os.environ["feature_data_output"], index=False)

        display(feature_data[:3])

    features = pd.read_parquet(os.environ["features_output"])
    future_prediction_targets_agg = pd.read_parquet(os.environ["future_prediction_targets_agg_output"])

    rank_features(
        features=features,
        future_prediction_targets_agg=future_prediction_targets_agg,
        feature_selection_subsample_start=FEATURE_SELECTION_SUBSAMPLE_START,
        feature_selection_subsample_end=FEATURE_SELECTION_SUBSAMPLE_END,
        feature_selection_subsample_amount=FEATURE_RANKING_SUBSAMPLE_AMOUNT,
        output_resolution_minutes=int(os.environ["OUTPUT_RESOLUTION"]),
    )

    

Correlation matrix

In [6]:
if False:
    def add_windows():
        HORIZONS_PER_WINDOW = int(os.environ["HORIZONS_PER_WINDOW"])
        OUTPUT_RESOLUTION = int(os.environ["OUTPUT_RESOLUTION"])

        _fd = pd.read_parquet(os.environ["feature_data_output"])
        horizon_cols = [c for c in _fd.columns if c.startswith("h") and c[1:].isdigit()]
        n_windows = len(horizon_cols) // HORIZONS_PER_WINDOW
        mi_df = _fd.set_index("feature")[horizon_cols]

        window_selections = {}
        window_mi_matrices = {}

        for w in range(n_windows):
            start_h = w * HORIZONS_PER_WINDOW
            end_h = start_h + HORIZONS_PER_WINDOW
            win_cols = horizon_cols[start_h:end_h]

            start_min = start_h * OUTPUT_RESOLUTION
            end_min = end_h * OUTPUT_RESOLUTION
            label = f"{start_min // 60}–{end_min // 60}h"

            win_mi = mi_df[win_cols]
            window_mi_matrices[label] = win_mi
            window_selections[label] = win_mi.mean(axis=1).sort_values(ascending=False).index.tolist()

        return window_selections, window_mi_matrices

    window_selections, window_mi_matrices = add_windows()

In [7]:
if False:
    def mi_matrix_facets(window_selections: dict, window_mi_matrices: dict, top_n: int = 20, ncols: int = 4):
        """All window MI heatmaps in one faceted figure with a shared colorbar."""
        labels = list(window_selections.keys())
        nrows  = (len(labels) + ncols - 1) // ncols

        _cmap = LinearSegmentedColormap.from_list(
            "mi_vibrant", ["#0a0a1a", "#2e1a6e", "#7b2d8b", "#d63a6e", "#f4845f"], N=256
        )

        # Pre-normalise each window's MI matrix, sorted by window MI, top_n features
        win_norms = {}
        for label, feats in window_selections.items():
            mi_win = window_mi_matrices[label].loc[feats]
            mi_win = mi_win.loc[mi_win.mean(axis=1).sort_values(ascending=False).index]
            win_norms[label] = (1 - np.exp(-mi_win)).head(top_n)

        col_w   = 9
        row_h   = max(3.5, top_n * 0.27)
        CBAR_H  = 0.1
        TITLE_H = 0.001   # extra space at top for the suptitle

        with plt.style.context("dark_background"):
            fig = plt.figure(figsize=(col_w * ncols, row_h * nrows + CBAR_H + TITLE_H))
            fig.patch.set_facecolor("#1e1e1e")

            gs = fig.add_gridspec(
                nrows + 1, ncols,
                height_ratios=[CBAR_H] + [row_h] * nrows,
                hspace=0.25, wspace=0.35,
                top=0.90,
            )
            cax = fig.add_subplot(gs[0, :])

            for idx, label in enumerate(labels):
                r = idx // ncols + 1
                c = idx % ncols
                ax = fig.add_subplot(gs[r, c])
                ax.set_facecolor("#1e1e1e")

                data       = win_norms[label]
                n_feat     = data.shape[0]
                annot_size = max(5, min(9, int(160 / max(n_feat, 1))))
                y_size     = max(6, min(9, int(200 / max(n_feat, 1))))

                sns.heatmap(
                    data, ax=ax, cmap=_cmap, vmin=0, vmax=1,
                    annot=True, fmt=".2f",
                    annot_kws={"size": annot_size, "color": "white"},
                    linewidths=0, linecolor="#2e2e2e", cbar=False,
                )
                ax.set_title(label, color="white", fontsize=10, pad=6)
                ax.xaxis.set_ticks_position("top")
                ax.xaxis.set_label_position("top")
                ax.tick_params(axis="x", labelsize=7, rotation=45, colors="white")
                ax.tick_params(axis="y", labelsize=y_size, rotation=0, colors="white")
                plt.setp(ax.get_xticklabels(), ha="left")

            # Shared colorbar in the top strip
            sm   = plt.cm.ScalarMappable(cmap=_cmap, norm=plt.Normalize(vmin=0, vmax=1))
            cbar = fig.colorbar(sm, cax=cax, orientation="horizontal")
            cax.xaxis.set_ticks_position("top")
            cax.xaxis.set_label_position("top")
            cbar.set_label("MI  (1 − e⁻ᴹᴵ)  —  normalised per feature", color="white", labelpad=6)
            cbar.ax.tick_params(colors="white")

            _window_hours = int(os.environ['HORIZONS_PER_WINDOW']) * int(os.environ['OUTPUT_RESOLUTION']) // 60
            fig.suptitle(
                f"Feature–target MI by {_window_hours}-hour window  (top {top_n} features per window)",
                color="white", fontsize=14, y=0.97,
            )
            plt.show()


    mi_matrix_facets(window_selections, window_mi_matrices, top_n=20, ncols=2)


Remove duplicate features (post-ranking)

In [8]:
if False:
    def remove_duplicate_features(feature_data):
        LINEAR_CORR_THRESHOLD = 0.95     # |Pearson| above this → linear duplicate
        NONLINEAR_CORR_THRESHOLD = 0.95  # |Spearman| above this → monotonic non-linear duplicate

        ranked_features = feature_data["feature"].tolist()  # already sorted best → worst by mean_mi

        # Load full feature matrix then subsample rows for correlation estimation
        features_full = pd.read_parquet(os.environ["features_output"])
        cols_present = [f for f in ranked_features if f in features_full.columns]
        features_full = features_full[cols_present]

        n_rows_total = len(features_full)
        n_rows = min(DEDUP_SUBSAMPLE_AMOUNT, n_rows_total)
        if n_rows < n_rows_total:
            rng = np.random.default_rng(42)
            subsample_idx = np.sort(rng.choice(n_rows_total, size=n_rows, replace=False))
            features_full = features_full.iloc[subsample_idx]

        n_rows, n_features = features_full.shape
        print(
            f"Standardising {n_rows:,} rows × {n_features} features "
            f"(subsample of {n_rows_total:,} total rows)...",
            flush=True,
        )

        # Standardise columns → Pearson corr = (Z.T @ Z) / n_rows per column pair
        col_means = features_full.mean()
        col_stds = features_full.std().replace(0, 1)
        Z_pearson = ((features_full - col_means) / col_stds).values.astype(np.float32)

        # Rank then standardise → Spearman via same dot product trick
        ranks = features_full.rank()
        rank_means = ranks.mean()
        rank_stds = ranks.std().replace(0, 1)
        Z_spearman = ((ranks - rank_means) / rank_stds).values.astype(np.float32)

        col_index = {col: i for i, col in enumerate(cols_present)}

        kept_indices = []  # column positions of kept features in Z_pearson / Z_spearman
        duplicate_flags = {}

        for feature in tqdm(ranked_features, desc="Deduplicating features", leave=True):
            if feature not in col_index:
                # feature absent from the loaded parquet — keep unconditionally
                duplicate_flags[feature] = False
                continue
            fi = col_index[feature]
            if not kept_indices:
                duplicate_flags[feature] = False
                kept_indices.append(fi)
                continue

            kept_arr = np.array(kept_indices)
            # Dot candidate column against each kept column → correlation vector; O(n_kept × n_rows)
            pearson_vals = np.abs(Z_pearson[:, fi] @ Z_pearson[:, kept_arr]) / n_rows
            spearman_vals = np.abs(Z_spearman[:, fi] @ Z_spearman[:, kept_arr]) / n_rows
            is_dup = bool(pearson_vals.max() > LINEAR_CORR_THRESHOLD or spearman_vals.max() > NONLINEAR_CORR_THRESHOLD)
            duplicate_flags[feature] = is_dup
            if not is_dup:
                kept_indices.append(fi)

        feature_data_flagged = feature_data.copy()
        feature_data_flagged["is_duplicate"] = feature_data_flagged["feature"].map(duplicate_flags)

        n_kept = feature_data_flagged["is_duplicate"].eq(False).sum()
        n_removed = feature_data_flagged["is_duplicate"].eq(True).sum()
        print(
            f"Kept {n_kept} unique features, flagged {n_removed} duplicates "
            f"(Pearson threshold={LINEAR_CORR_THRESHOLD}, Spearman threshold={NONLINEAR_CORR_THRESHOLD})",
            flush=True,
        )

        feature_data_unique = feature_data_flagged[feature_data_flagged["is_duplicate"] == False].reset_index(drop=True)

        feature_data_unique.to_parquet(os.environ["feature_data_unique_output"], index=False)
        display(feature_data_unique[:3])

        return feature_data_unique, feature_data_flagged, features_full


    feature_data = pd.read_parquet(os.environ["feature_data_output"])
    feature_data_unique, feature_data_flagged, features_subsampled = remove_duplicate_features(feature_data)


In [9]:
if False:
    import matplotlib.pyplot as plt
    from matplotlib.colors import LinearSegmentedColormap

    TOP_N = 20

    # Top N duplicate features by mean MI (highest MI duplicates — most "interesting" redundancies)
    duplicate_features = (
        feature_data_flagged[feature_data_flagged["is_duplicate"] == True]
        .head(TOP_N)["feature"]
        .tolist()
    )
    top_features = [f for f in duplicate_features if f in features_subsampled.columns]
    corr = features_subsampled[top_features].corr(method="pearson")

    # Diverging colormap with accent at 75th percentile (0.5 on the -1→1 scale = position 0.75)
    # blue → navy (mid) → electric cyan accent → crimson
    _cmap = LinearSegmentedColormap.from_list(
        "dark_div_accent",
        [
            (0.00, "#1565C0"),   # -1.0  deep blue
            (0.50, "#0D1B2A"),   #  0.0  dark navy (midpoint)
            (0.75, "#FFB300"),   # +0.5  electric cyan accent (75th percentile)
            (1.00, "#C62828"),   # +1.0  deep crimson
        ],
    )

    with plt.style.context("dark_background"):
        fig, ax = plt.subplots(figsize=(12, 10), facecolor="#1e1e1e")
        ax.set_facecolor("#1e1e1e")
        im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap=_cmap, aspect="auto")
        cbar = plt.colorbar(im, ax=ax, label="Pearson correlation")
        cbar.ax.yaxis.label.set_color("white")
        cbar.ax.tick_params(colors="white")
        ax.set_xticks(range(len(top_features)))
        ax.set_yticks(range(len(top_features)))
        ax.set_xticklabels(corr.columns, rotation=45, ha="right", fontsize=8, color="white")
        ax.set_yticklabels(corr.index, fontsize=8, color="white")
        ax.set_title(f"Pearson correlation — top {len(top_features)} duplicate features (by mean MI)", fontsize=11, color="white")
        ax.tick_params(colors="white")
        for spine in ax.spines.values():
            spine.set_edgecolor("#444444")
        plt.tight_layout()
        plt.show()


In [10]:
if False:
    import matplotlib.pyplot as plt

    df_plot = feature_data_flagged.sort_values("mean_mi", ascending=False).reset_index(drop=True)
    df_plot["rank_pos"] = range(1, len(df_plot) + 1)

    unique_mi = df_plot["mean_mi"].where(df_plot["is_duplicate"] == False, 0)
    dup_mi = df_plot["mean_mi"].where(df_plot["is_duplicate"] == True, 0)

    with plt.style.context("dark_background"):
        fig, ax = plt.subplots(figsize=(14, 5), facecolor="#1e1e1e")
        ax.set_facecolor("#1e1e1e")

        ax.fill_between(df_plot["rank_pos"], unique_mi, alpha=0.85, color="#1565C0", label="Unique")
        ax.fill_between(df_plot["rank_pos"], dup_mi, alpha=0.75, color="#C62828", label="Duplicate (removed)")

        ax.set_xlabel("Feature rank (most unique → least)", color="white", fontsize=10)
        ax.set_ylabel("Mean mutual information", color="white", fontsize=10)
        ax.set_title("Feature uniqueness by MI rank", fontsize=11, color="white")
        ax.tick_params(colors="white")
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.spines["bottom"].set_edgecolor("#444444")
        ax.spines["left"].set_edgecolor("#444444")
        ax.legend(facecolor="#2a2a2a", edgecolor="#444444", labelcolor="white")
        plt.tight_layout()
        plt.show()


Feature selection

In [11]:
if True:
    def feature_data_selection(
        feature_data,
        features,
        future_prediction_targets_agg,
        subsample_start: pd.Timestamp,
        subsample_end: pd.Timestamp,
        subsample_amount: int,
    ):
        # Number of equally-spaced k values to evaluate between 1 and n_available_features — tune this
        N_K_VALUES = 3
        # Number of equally-spaced horizons to evaluate — e.g. 10 tests every ~10th horizon; None = all
        N_HORIZON_STEPS = 5
        N_CV_SPLITS = 5   # time-series walk-forward folds
        # Thread count — each thread holds one X_h copy in the shared process heap (no worker spawning)
        N_PARALLEL_JOBS = min(_NCPU, 8)

        # Lightweight LightGBM for CV speed — not production params
        _LGBM_CV_PARAMS = {
            "objective": "regression_l1",
            "metric": "mae",
            "n_estimators": 500,       # high ceiling; early stopping exits well before this
            "learning_rate": 0.05,
            "num_leaves": 31,
            "min_child_samples": 20,
            "feature_fraction": 0.8,
            "bagging_fraction": 0.8,
            "bagging_freq": 5,
            "n_iter_no_change": 20,    # early stopping: halt if val MAE doesn't improve for 20 rounds
            "random_state": 42,
            "n_jobs": 1,               # parallelism handled externally by joblib
            "verbose": -1,
        }

        horizon_cols = [c for c in feature_data.columns if c.startswith("h") and c[1:].isdigit()]
        mi_matrix = feature_data.set_index("feature")[horizon_cols]  # (n_features, n_horizons)
        available_features = [f for f in mi_matrix.index if f in features.columns]  # Only keep features that exist in the in-memory features DataFrame

        # Time-range filter then random row subsample
        shared_idx = features.index.intersection(future_prediction_targets_agg.index)
        shared_idx = shared_idx[(shared_idx >= subsample_start) & (shared_idx <= subsample_end)]
        n_total = len(shared_idx)
        n_rows = min(subsample_amount, n_total)
        if n_rows < n_total:
            rng = np.random.default_rng(42)
            chosen = np.sort(rng.choice(n_total, size=n_rows, replace=False))
            shared_idx = shared_idx[chosen]
        print(f"CV subsample: {n_rows:,} rows (of {n_total:,} in range {subsample_start.date()} – {subsample_end.date()})", flush=True)

        # Arrays live in the main process; threads share them without copying (no pickling, no mmap needed)
        X_arr = features.loc[shared_idx, available_features].values.astype(np.float32)  # (n_rows, n_features)
        Y_arr = future_prediction_targets_agg.loc[shared_idx].values.astype(np.float32)  # (n_rows, n_horizons)
        mi_arr = mi_matrix.loc[available_features].values.astype(np.float32)             # (n_features, n_horizons)

        x_mb = X_arr.nbytes / 1e6
        print(f"X_arr: {X_arr.shape} ({x_mb:.0f} MB) | threads: {N_PARALLEL_JOBS}", flush=True)

        tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS)
        fold_indices = list(tscv.split(X_arr))  # pre-compute once

        # Equally-spaced grid from 1 → n_available_features, deduplicated and sorted
        k_grid = sorted(set(int(round(k)) for k in np.linspace(1, len(available_features), N_K_VALUES)))

        # Equally-spaced horizon indices to evaluate; None = all horizons
        n_h = len(horizon_cols)
        if N_HORIZON_STEPS is None or N_HORIZON_STEPS >= n_h:
            horizon_indices = list(range(n_h))
        else:
            horizon_indices = sorted(set(int(round(i)) for i in np.linspace(0, n_h - 1, N_HORIZON_STEPS)))

        n_tasks = len(horizon_indices) * len(k_grid) * len(fold_indices)
        print(
            f"Horizons: {len(horizon_indices)} of {n_h} | k grid: {k_grid} | folds: {len(fold_indices)} | "
            f"total tasks: {n_tasks}",
            flush=True,
        )

        # Per-fold worker — trains exactly one LightGBM model per call.
        # backend="threading": LightGBM fit() is pure C++ and releases the GIL, so threads get genuine
        # parallelism without spawning worker processes. Arrays are shared in the same heap — no pickling,
        # no memory-mapping, no worker heap accumulation between tasks.
        def _eval_horizon_k_fold(h_idx, k, fold_idx):
            import warnings
            warnings.filterwarnings("ignore", message="X does not have valid feature names", category=UserWarning)

            mi_scores = mi_arr[:, h_idx]
            top_k_idx = np.argsort(mi_scores)[::-1][:k]
            X_h = X_arr[:, top_k_idx]
            y = Y_arr[:, h_idx]

            train_idx, val_idx = fold_indices[fold_idx]
            y_tr, y_va = y[train_idx], y[val_idx]
            mask_tr = np.isfinite(y_tr)
            mask_va = np.isfinite(y_va)
            if mask_tr.sum() < 50 or mask_va.sum() < 10:
                return h_idx, k, fold_idx, None

            m = lgb.LGBMRegressor(**_LGBM_CV_PARAMS)
            m.fit(
                X_h[train_idx][mask_tr], y_tr[mask_tr],
                eval_set=[(X_h[val_idx][mask_va], y_va[mask_va])],
            )
            preds = m.predict(X_h[val_idx][mask_va])
            mae = float(np.mean(np.abs(y_va[mask_va] - preds)))
            del m, X_h
            return h_idx, k, fold_idx, mae

        tasks = [
            (h_idx, k, fold_idx)
            for h_idx in horizon_indices
            for k in k_grid
            for fold_idx in range(len(fold_indices))
        ]

        results_gen = Parallel(n_jobs=N_PARALLEL_JOBS, backend="threading", batch_size=1, return_as="generator_unordered")(
            delayed(_eval_horizon_k_fold)(h_idx, k, fold_idx)
            for h_idx, k, fold_idx in tasks
        )

        fold_mae_acc = {}
        for h_idx, k, fold_idx, mae in tqdm(results_gen, total=len(tasks), desc="CV feature search"):
            key = (h_idx, k)
            if key not in fold_mae_acc:
                fold_mae_acc[key] = []
            if mae is not None:
                fold_mae_acc[key].append(mae)

        cv_records = [
            {"horizon": horizon_cols[h_idx], "k": k, "cv_mae": float(np.mean(maes)) if maes else float("inf")}
            for (h_idx, k), maes in fold_mae_acc.items()
        ]

        cv_df = pd.DataFrame(cv_records)
        best_idx = cv_df.groupby("horizon")["cv_mae"].idxmin()
        horizon_best_k = cv_df.loc[best_idx].rename(columns={"k": "best_k"}).reset_index(drop=True)

        display(horizon_best_k)
        horizon_best_k.to_parquet(os.environ["cv_selection_output"], index=False)
        print(f"Saved CV selection → {os.environ['cv_selection_output']}", flush=True)

    feature_data_unique = pd.read_parquet(os.environ["feature_data_unique_output"])
    features = pd.read_parquet(os.environ["features_output"])
    future_prediction_targets_agg = pd.read_parquet(os.environ["future_prediction_targets_agg_output"])

    feature_data_selection(
        feature_data_unique,
        features,
        future_prediction_targets_agg,
        subsample_start=FEATURE_SELECTION_CV_SUBSAMPLE_START,
        subsample_end=FEATURE_SELECTION_CV_SUBSAMPLE_END,
        subsample_amount=FEATURE_SELECTION_CV_SUBSAMPLE_AMOUNT,
    )


CV subsample: 50,000 rows (of 420,769 in range 2019-01-01 – 2023-01-01)
X_arr: (50000, 438) (88 MB) | threads: 8
Horizons: 5 of 96 | k grid: [1, 220, 438] | folds: 5 | total tasks: 75


CV feature search: 100%|██████████| 75/75 [00:50<00:00,  1.50it/s]


,horizon,best_k,cv_mae
0,h1,438,21.813574
1,h25,220,39.178601
2,h49,220,38.892499
3,h72,220,47.356081
4,h96,220,42.478688


Saved CV selection → ../4_Features select/Feature_store/NSW_cv_selection_1_dispatch_price.parquet


Feature selection output

In [12]:
if True:
    def feature_selection_output():
        # Load CV results (best k per horizon) and MI scores
        horizon_best_k = pd.read_parquet(os.environ["cv_selection_output"])
        feature_data_unique = pd.read_parquet(os.environ["feature_data_unique_output"])

        horizon_cols = [c for c in feature_data_unique.columns if c.startswith("h") and c[1:].isdigit()]
        mi_matrix = feature_data_unique.set_index("feature")[horizon_cols]  # (n_features, n_horizons)

        # For each evaluated horizon, select the top-best_k features by MI score.
        # Horizons not evaluated by CV (subsampled out) carry no selection row → column stays False.
        selection = pd.DataFrame(False, index=mi_matrix.index, columns=horizon_cols)

        for _, row in horizon_best_k.iterrows():
            h = row["horizon"]
            k = int(row["best_k"])
            if h not in mi_matrix.columns:
                continue
            top_k_features = mi_matrix[h].nlargest(k).index
            selection.loc[top_k_features, h] = True

        n_any = selection.any(axis=1).sum()
        print(
            f"Selected features per horizon | "
            f"features in any horizon: {n_any} of {len(mi_matrix)} | "
            f"per-horizon range: {selection.sum().min()}–{selection.sum().max()}",
            flush=True,
        )

        selection.to_parquet(os.environ["selected_features_output"])
        display(selection[:3])
    
    feature_selection_output()


Selected features per horizon | features in any horizon: 438 of 438 | per-horizon range: 0–438


,h1,h2,h3,h4,h5,h6,h7,h8,h9,h10,h11,h12,h13,h14,h15,h16,h17,h18,h19,h20,h21,h22,h23,h24,h25,h26,h27,h28,h29,h30,h31,h32,h33,h34,h35,h36,h37,h38,h39,h40,h41,h42,h43,h44,h45,h46,h47,h48,h49,h50,h51,h52,h53,h54,h55,h56,h57,h58,h59,h60,h61,h62,h63,h64,h65,h66,h67,h68,h69,h70,h71,h72,h73,h74,h75,h76,h77,h78,h79,h80,h81,h82,h83,h84,h85,h86,h87,h88,h89,h90,h91,h92,h93,h94,h95,h96
feature,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
nsw_price_q90_336,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
nsw_price_asinh_rmean_336,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
nsw_price_rmean_2016,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True
